In [1]:
import pandas as pd
import sys
import IPython
import os
import numpy as np
from pathlib import Path
from datetime import date, timedelta
import QuantLib as ql
from datetime import date, datetime

ipynb_path = Path(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"]).resolve()
notebook_name = "/".join(str(ipynb_path).split("/")[-5:])
notebook_dir = os.path.dirname(notebook_name)
passiv_dir = Path(notebook_dir).parent

myg_root = ipynb_path.parents[2]
python_root = ipynb_path.parents[3]
for p in (myg_root, python_root):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from _passiv.libraries import passiv_funktionen
from _passiv.libraries import passiv_import_data
from _passiv.libraries import passiv_kupons
from _passiv.libraries import passiv_rlz
from _passiv.libraries import passiv_bond_values

bm_dir = Path(passiv_dir, 'bm_files')

In [2]:
bm = passiv_import_data.import_data(verbose=True, warning=False)
#bm.query("country=='IT'")

...found: 1 file[s] with 20260806 as date...
File: Q:\Rates\python\myg\_passiv\bm_files\jpm\FilePolling.32190.COMP_GBROAD_20260806.csv
Date in Filename: 20260806
Using 'weight_%_daily_usd_rtrn' as weight (NOT 'weight_%_daily_stats')
date format: %d-%b-%Y
Date: 2026-08-06
Provider: jpm
Key Rates: ['krd03', 'krd05', 'krd07', 'krd10', 'krd15', 'krd30', 'krd50']
Maturity Buckets: ['1-3', '3-5', '5-7', '7-10', '10+']
Countries: ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'GR', 'IE', 'IT', 'NL', 'PT', 'SI', 'SK']
Number of Bonds: 491


In [3]:
benchmark_settlement_date = '2026-08-10'
bm2 = bm.copy()
bm2['first_coupon_type'] = (bm2.apply(lambda row: passiv_bond_values.deduce_coupon_type(settlement_date=benchmark_settlement_date, 
                                                                                              maturity_date=row['maturity'], 
                                                                                              coupon=row['coupon'], yld=row['yield'], 
                                                                                              freq=row['freq'], 
                                                                                              target_mac_dur=row['mac_dur'], 
                                                                                              target_price=row['price'], 
                                                                                              issue_date=row['issue_date'], 
                                                                                              compounding='annual',
                                                                                              discount_method='standard'), axis=1))


In [36]:

valuation_date = '2026-08-15'

bm2[['mac_dur_new', 'price_new']] = (bm2.apply(lambda row: passiv_bond_values.calculate_macaulay_duration(settlement_date=valuation_date,
                                                                                       maturity_date=row['maturity'],
                                                                                       coupon=row['coupon']/100, 
                                                                                       yld=row['yield']/100,
                                                                                       freq=row['freq'],
                                                                                       first_coupon_type=row['first_coupon_type'],
                                                                                       issue_date=row['issue_date'],
                                                                                       compounding='annual',
                                                                                       discount_method='standard'), axis=1,result_type='expand'))
